# Lasso Regression — Optimization

**Goal.** Build practical algorithms that solve the Lasso problem (1.2) of `02_mathematics.ipynb`, given that no closed form exists. Two algorithm families:

1. **Coordinate descent** — update one coordinate at a time, each update is a soft-thresholding step (Theorem 4.2 of `02_mathematics.ipynb`). The textbook Lasso algorithm; this is what `sklearn.linear_model.Lasso` uses under the hood.
2. **Proximal gradient (ISTA / FISTA)** — full-gradient on the smooth part, then a soft-thresholding step on the L1 part. Cleaner theory, GPU-friendly, easy to extend to other regularisers.

Both algorithms boil down to repeated soft-thresholding. The difference is how each builds the input z.

**Role of this notebook.** Algorithms — pseudocode plus minimal demo code. Math is in `02_mathematics.ipynb`; full implementation in `05_hands_on_programming.ipynb`.

**Prerequisites.** `02_mathematics.ipynb` (the L1 loss, the first-order conditions, the soft-thresholding formula). `01_linear_regression/03_optimization.ipynb` (gradient descent and Lipschitz smoothness — re-used for ISTA).

**Stage map.** `01_intuition` → `02_mathematics` → **`03_optimization`** → `04_statistics` → `05_hands_on_programming`.

**Five questions.**

1. Why does plain gradient descent struggle on Lasso?
2. What is the coordinate-descent update for Lasso, and why is each step a closed-form soft-thresholding?
3. What is the proximal-gradient (ISTA) algorithm, and how does FISTA accelerate it?
4. How do the two algorithms compare in practice?
5. How do we sweep $\lambda$ efficiently?

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import random

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

plt.rcParams["figure.dpi"] = 90

## 1. Why plain GD does not fit cleanly

Recall `01_linear_regression/03_optimization.ipynb` Algorithm 1: `$\theta_{k+1}$ = $\theta_k$ - $\eta$ $\cdot$ $\nabla L$($\theta_k$)`. This relied on `$\nabla L$` *existing*. For Lasso, the gradient does not exist at any point with a zero coordinate (`02_mathematics.ipynb` §2), and we expect the optimum to have many zero coordinates — so we cannot just plug in.

Two workarounds in principle:

1. **Subgradient method.** Replace the gradient by *any* element of the subdifferential and take the step. It converges, but at rate $O(1 / \sqrt{k})$ rather than the `O(1/k)` of smooth GD. Slow.
2. **Smoothing.** Replace `$|\theta_j|$` by a smooth approximation like $\sqrt{\theta_j^2 + \varepsilon}$. Works, but loses the exact-zero property — the whole point of Lasso. Bad trade.

The right approach is to exploit the **structure** of the loss: the data term is smooth and convex, the penalty term is non-smooth but *separable* and has a *closed-form* one-dimensional optimisation (soft-thresholding). Coordinate descent and proximal-gradient methods both lean on this split.

## 2. Coordinate descent — the textbook Lasso algorithm

### 2.1 Idea

Pick one coordinate j at a time. Hold the other p - 1 coordinates fixed. The Lasso problem in that single coordinate is a **one-dimensional Lasso**, whose closed form is the soft-thresholding (Theorem 4.2 of `02_mathematics.ipynb`). Cycle through j = 1, 2, …, p, p, p - 1, …, 1, 2, … until convergence.

### 2.2 Derivation of the coordinate update

Fix j. Write the Lasso loss as a function of $\theta_j$ only, with the other $\theta_k$ for $k \neq j$ treated as constants. Let

> $r_{-j} := y - \sum_{k \neq j} x_k \theta_k$

be the **partial residual** — the residual we would have if we set $\theta_j$ to zero. Then

```
(1/n) $\cdot$ $\|X \theta - y\|_2^2$   =   (1/n) $\cdot$ $\|x_j\theta_j - r_{-j}\|_2^2$
                          =   (1/n) $\cdot$ ( $\|x_j\|_2^2$ $\cdot$ $\theta_j$^2   -   2 $\cdot$ ⟨$x_j$, r_{-j}⟩ $\cdot$ $\theta_j$   +   constant )
```

("constant" is whatever does not depend on $\theta_j$). Adding the penalty `$\lambda$ $\cdot$ $|\theta_j|$ + (constant)`, the j-th coordinate problem is

```
argmin_{$\theta_j$}   (1/n) $\cdot$ $\|x_j\|_2^2$ $\cdot$ $\theta_j$^2   -   (2/n) $\cdot$ ⟨$x_j$, r_{-j}⟩ $\cdot$ $\theta_j$   +   $\lambda$ $\cdot$ $|\theta_j|$.
```

Complete the square (divide through by `(2/n) $\cdot$ $\|x_j\|_2^2$`):

```
= (1/n) $\cdot$ $\|x_j\|_2^2$ $\cdot$ ( $\theta_j$  -  ⟨$x_j$, r_{-j}⟩ / $\|x_j\|_2^2$ )^2   +   $\lambda$ $\cdot$ $|\theta_j|$   +   constant.
```

This is now Theorem 4.2 of `02_mathematics.ipynb` with `z := ⟨$x_j$, r_{-j}⟩ / $\|x_j\|_2^2$` and threshold `$\lambda$ $\cdot$ n / (2 $\cdot$ $\|x_j\|_2^2$)`. Therefore

> $\theta_{j,\mathrm{new}} = S_{\lambda n/(2\|x_j\|_2^2)}\left(\langle x_j, r_{-j} \rangle / \|x_j\|_2^2\right).$   (2.1)

where `S_t(z) := $\text{sign}(z)$ $\cdot$ max(|z| - t, 0)` is the soft-thresholding operator.

### 2.3 Algorithm

```
ALGORITHM:  Coordinate Descent for Lasso

Input:    X $\in$ R^{n $\times$ p} with columns x_1, …, x_p,  y $\in$ R^n,  $\lambda$ > 0,
          initial $\theta_{0}$,  tolerance tol,  max sweeps K.
Output:   $\theta_{K}$ $\approx$ argmin  $L_{\text{lasso}}(\theta)$.

Pre-compute:  c_j $\leftarrow$ $\|$x_j$\|_{2}$^2   for j = 1, …, p
              r   $\leftarrow$ y - $X \theta$      (current full residual)

for k = 1, …, K:
    for j = 1, …, p:
        # Use full residual to recover the partial residual cheaply.
        # r_{-j} = r + x_j $\cdot$ $\theta_{j}$
        z_j      $\leftarrow$  ( ⟨x_j, r⟩  +  c_j $\cdot$ $\theta_{j}$ ) / c_j
        $\theta_{j,\mathrm{new}}$  $\leftarrow$  soft_thresh(z_j,  n $\lambda$ / (2 c_j))
        r        $\leftarrow$  r  +  x_j $\cdot$ ($\theta_{j}$ - $\theta_{j,\mathrm{new}}$)
        $\theta_{j}$      $\leftarrow$  $\theta_{j,\mathrm{new}}$
    if  max coordinate change  <  tol:  break
return $\theta$
```

**Why this is the right algorithm.**

- Each inner step costs `Θ(n)`: one inner product `⟨$x_j$, r⟩`, one scalar update of `r`. A full sweep over p coordinates is `Θ(n p)`. The same as one step of full-gradient.
- The full residual `r` is maintained incrementally, never recomputed from scratch.
- Each update is **exact** for the one-dimensional sub-problem. So coordinate descent on Lasso descends *strictly* monotonically (a non-trivial property — coordinate descent for non-smooth objectives can fail in general, but the separability of the L1 penalty makes Lasso a friendly case).
- Empirically, coordinate descent converges in 10–50 sweeps for typical Lasso problems.

### 2.4 Minimal coordinate-descent demo

Implement Algorithm 2.3 directly. Run on a small synthetic Lasso problem and compare with `sklearn.linear_model.Lasso`.

In [ ]:
def soft_thresh(z, t):
    """Soft-thresholding: S_t(z) = sign(z) · max(|z| − t, 0)."""
    return np.sign(z) * np.maximum(np.abs(z) - t, 0.0)

def lasso_coord_descent(X, y, lam, n_sweeps=200, tol=1e-6):
    """Coordinate descent for L(θ) = (1/n)·‖Xθ − y‖² + λ·‖θ‖₁ on centred X, y."""
    n, p = X.shape
    c = (X * X).sum(axis=0)                         # c_j = ‖x_j‖²
    theta = np.zeros(p)
    r = y.copy()
    for sweep in range(n_sweeps):
        max_change = 0.0
        for j in range(p):
            z_j = (X[:, j] @ r + c[j] * theta[j]) / c[j]
            new = soft_thresh(z_j, n * lam / (2 * c[j]))
            delta = new - theta[j]
            if delta != 0:
                r -= X[:, j] * delta
                theta[j] = new
                if abs(delta) > max_change: max_change = abs(delta)
        if max_change < tol:
            break
    return theta, sweep + 1

# Sparse synthetic problem.
n, p = 200, 30
X = rng.normal(size=(n, p))
true_theta = np.zeros(p); true_theta[:5] = [3, -2, 1.5, 1, -0.8]
y = X @ true_theta + rng.normal(0, 0.5, size=n)
X = X - X.mean(axis=0); y = y - y.mean()           # centring (no intercept)

lam = 0.05
theta_ours, n_iter = lasso_coord_descent(X, y, lam)
theta_skl = Lasso(alpha=lam, fit_intercept=False, max_iter=20000).fit(X, y).coef_

print(f"converged in {n_iter} sweeps")
print(f"max |ours − sklearn|     = {np.max(np.abs(theta_ours - theta_skl)):.2e}")
print(f"ours: # non-zero = {int(np.sum(np.abs(theta_ours) > 1e-8))}")
print(f"true: # non-zero = 5")

## 3. Proximal gradient (ISTA)

Coordinate descent updates *one coordinate at a time*. Proximal gradient does the opposite — a *full-vector* gradient step on the smooth part of the loss, followed by a *single* soft-thresholding operation on all coordinates at once. Cleaner formulation, more general (the soft-threshold is just one example of a *proximal operator*), and naturally vectorisable on GPU.

### 3.1 The ISTA update

Split the loss as

> $L_{\text{lasso}}(\theta)$  =  $f(\theta)$  +  $g(\theta)$,    with   $f(\theta)$ := (1/n) $\cdot$ $\|X \theta - y\|_2^2$   and   $g(\theta)$ := $\lambda$ $\cdot$ $\|\theta\|_1$.

f is smooth with gradient $\nabla f(\theta) = (2/n)X^\top(X\theta - y)$. The **iterative soft-thresholding algorithm (ISTA)** is:

```
ALGORITHM:  ISTA (proximal gradient for Lasso)

Input:   X, y, $\lambda$, step size $\eta$ > 0, initial $\theta_{0}$, max iterations K.

for k = 0, 1, …, K - 1:
    g_k       $\leftarrow$  (2/n) $\cdot$ $X^T$ (X $\theta_k$ - y)               # smooth-part gradient
    z_k       $\leftarrow$  $\theta_k$  -  $\eta$ $\cdot$ g_k                       # full GD step on f
    $\theta_{k+1}$   $\leftarrow$  soft_thresh(z_k,  $\eta$ $\cdot$ $\lambda$)              # proximal step on g
return $\theta_{K}$
```

### 3.2 Why the soft-threshold appears

The general proximal-gradient framework solves `argmin (f + g)` by alternating a gradient step on `f` and a **proximal operator** on `g`. The proximal operator of `t $\cdot$ |z|` (acting coordinate-wise on a vector `z`) is exactly the soft-thresholding `S_t(z)` (Theorem 4.2 of `02_mathematics.ipynb`). Substituting gives the ISTA step above.

### 3.3 Convergence rate

Let $L_{\text{smooth}} = \lambda_{\max}(2X^\top X/n)$ be the Lipschitz constant of $\nabla f$. With step size $\eta = 1/L_{\text{smooth}}$, ISTA satisfies

> $L_{\text{lasso}}(\theta_k)$ - $L_{\text{lasso}}(\theta^*)$   $\le$   $\|\theta_0 - \theta^*\|_2^2$ / (2 $\eta$ $\cdot$ k)   =   O(1 / k).

(Beck & Teboulle 2009, *A Fast Iterative Shrinkage-Thresholding Algorithm*, SIAM J. Imaging Sci.)

The same O(1/k) sub-linear rate as smooth gradient descent on a convex problem (Theorem 4.2 of `01_linear_regression/03_optimization.ipynb`). The Lasso non-smoothness is absorbed *exactly* by the proximal step — no slowdown.

### 3.4 Demo: ISTA converges to the same answer as coordinate descent

In [ ]:
def lasso_ista(X, y, lam, n_iter=2000, eta=None):
    n, p = X.shape
    if eta is None:
        # Lipschitz constant of ∇f is λ_max(2 X^T X / n) = (2/n) · σ_max(X)².
        sigma_max = np.linalg.svd(X, compute_uv=False)[0]
        L = (2.0 / n) * sigma_max ** 2
        eta = 1.0 / L
    theta = np.zeros(p)
    losses = []
    for _ in range(n_iter):
        grad = (2.0 / n) * X.T @ (X @ theta - y)
        theta = soft_thresh(theta - eta * grad, eta * lam)
        losses.append(float(np.mean((X @ theta - y) ** 2) + lam * np.sum(np.abs(theta))))
    return theta, np.array(losses)

theta_ista, ista_losses = lasso_ista(X, y, lam=lam, n_iter=1500)
print(f"max |ours ISTA − sklearn|         = {np.max(np.abs(theta_ista - theta_skl)):.2e}")
print(f"max |ours ISTA − coord descent|   = {np.max(np.abs(theta_ista - theta_ours)):.2e}")
print(f"ISTA non-zero coefficients        = {int(np.sum(np.abs(theta_ista) > 1e-6))}")

## 4. FISTA — Nesterov-accelerated ISTA

ISTA's O(1/k) rate is fine but not optimal. **FISTA** (Fast ISTA, Beck & Teboulle 2009) achieves O(1/k^2) — the same speedup as Nesterov's momentum on smooth convex problems — by computing the gradient at an *extrapolated* point rather than the current iterate.

```
ALGORITHM:  FISTA

Initialise:  $\theta_{0}$ = $\theta_{-1}$ = 0, t_1 = 1

for k = 1, 2, …:
    y_k       $\leftarrow$  $\theta_{k-1}$  +  (($t_{k-1}$ - 1) / t_k) $\cdot$ ($\theta_{k-1}$ - $\theta_{k-2}$)    # extrapolation
    z_k       $\leftarrow$  y_k  -  $\eta$ $\cdot$ $\nabla f(y_k)$                                         # gradient at y_k
    $\theta_k$       $\leftarrow$  soft_thresh(z_k,  $\eta$ $\cdot$ $\lambda$)                                    # proximal step
    t_{k+1}   $\leftarrow$  ( 1 + $\sqrt{1 + 4 t_k^2}$ ) / 2                                  # update momentum scalar
```

**Cost per step.** Same as ISTA — one matrix-vector product. The extrapolation step is just two vector additions.

**Practical note.** FISTA is the default for *non-coordinate* Lasso solvers (e.g. when X is too large for column-wise access, or when working on GPU). Coordinate descent is faster when X is dense and column-accessible. Both are used in practice.

In [ ]:
def lasso_fista(X, y, lam, n_iter=2000, eta=None):
    n, p = X.shape
    if eta is None:
        sigma_max = np.linalg.svd(X, compute_uv=False)[0]
        eta = 1.0 / ((2.0 / n) * sigma_max ** 2)
    theta = np.zeros(p)
    theta_prev = theta.copy()
    t = 1.0
    losses = []
    for _ in range(n_iter):
        t_new = (1 + np.sqrt(1 + 4 * t * t)) / 2
        y_k = theta + ((t - 1) / t_new) * (theta - theta_prev)
        grad = (2.0 / n) * X.T @ (X @ y_k - y)
        theta_prev = theta
        theta = soft_thresh(y_k - eta * grad, eta * lam)
        t = t_new
        losses.append(float(np.mean((X @ theta - y) ** 2) + lam * np.sum(np.abs(theta))))
    return theta, np.array(losses)

theta_fista, fista_losses = lasso_fista(X, y, lam=lam, n_iter=1500)

# Compare convergence: how fast does each method approach the optimum?
loss_star = min(ista_losses.min(), fista_losses.min())
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(np.maximum(ista_losses - loss_star, 1e-16),  color="steelblue", label="ISTA")
ax.plot(np.maximum(fista_losses - loss_star, 1e-16), color="crimson",   label="FISTA")
ax.set_yscale("log")
ax.set_xlabel("iteration k")
ax.set_ylabel("L(θ_k) − L\\*  (log)")
ax.set_title("FISTA's O(1/k²) is visibly faster than ISTA's O(1/k)")
ax.legend()
plt.show()

print(f"max |ISTA  − sklearn|  = {np.max(np.abs(theta_ista  - theta_skl)):.2e}")
print(f"max |FISTA − sklearn|  = {np.max(np.abs(theta_fista - theta_skl)):.2e}")

## 5. The $\lambda$-path with warm starts

Cross-validation needs the fit at many $\lambda$ values. Three observations make a path solver much cheaper than M independent fits:

1. **Order matters.** Start from the *largest* $\lambda$ (where most coefficients are zero) and decrease. The active set grows slowly.
2. **Warm-start.** The solution at $\lambda_{k+1}$ is close to the solution at $\lambda_{k}$. Initialise from the previous answer, run only a few sweeps.
3. **Largest $\lambda$ has a closed form.** When $\lambda$ exceeds `$\lambda_{\max}$ := (2 / n) $\cdot$ max_j |$x_j$ᵀ y|`, the entire solution is $\theta$ = 0. Above this threshold no coordinate satisfies the activation condition. So start the path at $\lambda_{\max}$ and go down.

These three tricks are what make `sklearn.linear_model.lasso_path` and the GLMNET package fast in practice — typically 10–50 ms for an entire 100-point path on a moderate dataset.

## Takeaway

- **GD does not work directly** on Lasso because the gradient does not exist at the typical optimum. Subgradient methods work but are slow.
- **Coordinate descent (§2)** updates one $\theta_j$ at a time. Each update is a closed-form soft-thresholding (eq. 2.1). Cycles converge in 10–50 sweeps; this is what scikit-learn uses.
- **ISTA (§3)** does a full GD step on the data term, then a soft-threshold on all coordinates. Same `O(1/k)` rate as smooth GD; nice general framework.
- **FISTA (§4)** accelerates ISTA with Nesterov momentum to `O(1/k^2)`. Same per-step cost.
- **Path solver (§5).** Warm-start from $\lambda_{\max}$ = (2/n) $\cdot$ max_j |$x_j$ᵀy| (where $\hat{\theta}$ = 0) and decrease — typical cross-validation paths run in milliseconds.

Next: `04_statistics.ipynb` examines what $\hat{\theta}_{lasso}$ looks like as a random vector — bias, the **active set**, when Lasso recovers the true sparse support, and how to pick $\lambda$ in practice.